# 🧠 Pengaruh Tidur, Stres, dan Aktivitas Fisik pada Random Forest–SHAP untuk Prediksi Stroke
**Dataset:** NHANES 2015–2016  
**Metode:** Decision Tree, Random Forest, XGBoost + SHAP  
**Tugas Besar Kecerdasan Buatan — 2026**

---
## Alur Notebook
1. Install & Import Library
2. Download & Load Dataset NHANES
3. Merge Semua File by SEQN
4. Seleksi Kolom & Preprocessing
5. Desain Eksperimen (5 skenario)
6. Training & Evaluasi Model
7. Visualisasi Hasil
8. Analisis SHAP

## 📦 1. Install & Import Library

In [ ]:
# Install library tambahan yang tidak ada di Colab by default
!pip install shap imbalanced-learn xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import shap

# Styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
COLORS = ['#2C3E93', '#F5A623', '#10B981', '#E74C3C', '#8B5CF6']

print('✅ Semua library berhasil diimport!')

## 📥 2. Download & Load Dataset NHANES 2015–2016

In [ ]:
BASE_URL = 'https://wwwn.cdc.gov/nchs/nhanes/2015-2016/'

files = {
    'demo': 'DEMO_I.XPT',   # Demografi
    'mcq':  'MCQ_I.XPT',    # Medical Conditions (target: stroke)
    'bpq':  'BPQ_I.XPT',    # Blood Pressure (hipertensi)
    'diq':  'DIQ_I.XPT',    # Diabetes
    'bmx':  'BMX_I.XPT',    # Body Measurements (BMI, lingkar pinggang)
    'bpx':  'BPX_I.XPT',    # Blood Pressure Exam (tekanan darah)
    'smq':  'SMQ_I.XPT',    # Smoking
    'slq':  'SLQ_I.XPT',    # Sleep (NOVELTY 1)
    'dpq':  'DPQ_I.XPT',    # Depression/Stress PHQ-9 (NOVELTY 2)
    'paq':  'PAQ_I.XPT',    # Physical Activity (NOVELTY 3)
}

dfs = {}
for name, filename in files.items():
    url = BASE_URL + filename
    print(f'Downloading {filename}...', end=' ')
    try:
        dfs[name] = pd.read_sas(url, format='xport', encoding='utf-8')
        print(f'✅ {dfs[name].shape[0]} baris')
    except Exception as e:
        print(f'❌ Error: {e}')

print('\n✅ Semua file berhasil didownload!')

## 🔗 3. Merge Semua File by SEQN

In [ ]:
# Pilih kolom relevan dari setiap file
# ─── DEMOGRAFI ───────────────────────────────────────────
demo_cols = ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR']
demo_clean = dfs['demo'][demo_cols].copy()

# ─── TARGET: STROKE ──────────────────────────────────────
# MCQ160F: 1=Ya pernah stroke, 2=Tidak
mcq_cols = ['SEQN', 'MCQ160F', 'MCQ160B', 'MCQ160C', 'MCQ160E']
mcq_clean = dfs['mcq'][mcq_cols].copy()

# ─── HIPERTENSI & DIABETES ───────────────────────────────
bpq_cols = ['SEQN', 'BPQ020']
bpq_clean = dfs['bpq'][bpq_cols].copy()

diq_cols = ['SEQN', 'DIQ010']
diq_clean = dfs['diq'][diq_cols].copy()

# ─── PEMERIKSAAN FISIK ───────────────────────────────────
bmx_cols = ['SEQN', 'BMXBMI', 'BMXWC']
bmx_clean = dfs['bmx'][bmx_cols].copy()

bpx_cols = ['SEQN', 'BPXSY1', 'BPXDI1']
bpx_clean = dfs['bpx'][bpx_cols].copy()

# ─── MEROKOK ─────────────────────────────────────────────
smq_cols = ['SEQN', 'SMQ020', 'SMQ040']
smq_clean = dfs['smq'][smq_cols].copy()

# ─── TIDUR (NOVELTY 1) ───────────────────────────────────
slq_cols = ['SEQN', 'SLD012', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']
slq_clean = dfs['slq'][slq_cols].copy()

# ─── STRES / PHQ-9 (NOVELTY 2) ───────────────────────────
dpq_cols = ['SEQN'] + [f'DPQ0{i}0' for i in range(1, 10)]
dpq_clean = dfs['dpq'][dpq_cols].copy()
# Buat stress_score = total PHQ-9
phq_items = [f'DPQ0{i}0' for i in range(1, 10)]
dpq_clean['stress_score'] = dpq_clean[phq_items].apply(
    lambda row: row[row <= 3].sum() if row.notna().sum() >= 7 else np.nan, axis=1
)
dpq_clean = dpq_clean[['SEQN', 'stress_score']]

# ─── AKTIVITAS FISIK (NOVELTY 3) ─────────────────────────
paq_cols = ['SEQN', 'PAQ605', 'PAD615', 'PAQ650', 'PAQ665', 'PAD680']
paq_clean = dfs['paq'][paq_cols].copy()

# ─── MERGE SEMUA ─────────────────────────────────────────
df = demo_clean
for other in [mcq_clean, bpq_clean, diq_clean, bmx_clean,
              bpx_clean, smq_clean, slq_clean, dpq_clean, paq_clean]:
    df = df.merge(other, on='SEQN', how='left')

print(f'Total baris setelah merge: {df.shape[0]:,}')
print(f'Total kolom: {df.shape[1]}')
df.head()

## 🧹 4. Preprocessing

In [ ]:
# ─── BUAT TARGET VARIABEL ────────────────────────────────
# MCQ160F: 1=pernah stroke, 2=tidak → ubah ke 1/0
df['stroke'] = df['MCQ160F'].map({1.0: 1, 2.0: 0})

# Hapus baris yang tidak punya label stroke
df = df[df['stroke'].notna()].copy()
print(f'Distribusi stroke:\n{df["stroke"].value_counts()}')
print(f'Rasio: {df["stroke"].mean():.2%} positif stroke')

# Filter usia dewasa (18+)
df = df[df['RIDAGEYR'] >= 18].copy()
print(f'\nSetelah filter usia 18+: {len(df):,} baris')

In [ ]:
# ─── ENCODING VARIABEL KATEGORIK ─────────────────────────
# Konversi kode 1/2 menjadi 1/0 untuk variabel binary
binary_cols = {
    'BPQ020': 'hypertension',    # 1=Ya, 2=Tidak
    'DIQ010': 'diabetes',         # 1=Ya, 2=Tidak (3=Borderline → 1)
    'MCQ160B': 'heart_failure',
    'MCQ160C': 'coronary_disease',
    'MCQ160E': 'heart_attack',
    'SMQ020': 'ever_smoked',
    'SLQ050': 'sleep_problem_doctor',
    'PAQ605': 'vigorous_work',
    'PAQ650': 'vigorous_leisure',
    'PAQ665': 'moderate_leisure',
}
for old, new in binary_cols.items():
    if old in df.columns:
        df[new] = df[old].map({1.0: 1, 2.0: 0, 3.0: 1})  # borderline diabetes → 1

# Rename kolom numerik
rename_map = {
    'RIDAGEYR': 'age',
    'RIAGENDR': 'gender',          # 1=Male, 2=Female
    'RIDRETH3': 'race',
    'DMDEDUC2': 'education',
    'INDFMPIR': 'income_ratio',
    'BMXBMI':   'bmi',
    'BMXWC':    'waist_circ',
    'BPXSY1':   'systolic_bp',
    'BPXDI1':   'diastolic_bp',
    'SMQ040':   'current_smoker',  # 1=setiap hari, 2=kadang, 3=tidak
    'SLD012':   'sleep_hours',
    'SLQ030':   'snoring_freq',
    'SLQ040':   'sleep_apnea',
    'SLQ120':   'daytime_sleepy',
    'PAD615':   'vigorous_work_min',
    'PAD680':   'sedentary_min',
}
df = df.rename(columns=rename_map)

# Gender: 1=Male → 1, 2=Female → 0
df['gender'] = df['gender'].map({1.0: 1, 2.0: 0})

print('✅ Encoding selesai')

In [ ]:
# ─── DEFINISI KELOMPOK FITUR ─────────────────────────────

# Fitur klinis standar (BASELINE)
CLINICAL = [
    'age', 'gender', 'race', 'education', 'income_ratio',
    'bmi', 'waist_circ', 'systolic_bp', 'diastolic_bp',
    'hypertension', 'diabetes', 'heart_failure',
    'coronary_disease', 'heart_attack',
    'ever_smoked', 'current_smoker'
]

# Fitur tidur (NOVELTY 1)
SLEEP = ['sleep_hours', 'snoring_freq', 'sleep_apnea',
         'sleep_problem_doctor', 'daytime_sleepy']

# Fitur stres (NOVELTY 2)
STRESS = ['stress_score']

# Fitur aktivitas fisik (NOVELTY 3)
PHYSICAL = ['vigorous_work', 'vigorous_work_min', 'vigorous_leisure',
            'moderate_leisure', 'sedentary_min']

TARGET = 'stroke'

# Semua kolom yang dipakai
ALL_COLS = CLINICAL + SLEEP + STRESS + PHYSICAL + [TARGET]
df_model = df[ALL_COLS].copy()

print(f'Total fitur: {len(ALL_COLS) - 1}')
print(f'  - Klinis  : {len(CLINICAL)}')
print(f'  - Tidur   : {len(SLEEP)}')
print(f'  - Stres   : {len(STRESS)}')
print(f'  - Aktivitas: {len(PHYSICAL)}')

In [ ]:
# ─── HANDLING MISSING VALUES ─────────────────────────────
print('Missing values sebelum:')
print(df_model.isnull().sum()[df_model.isnull().sum() > 0])

# Drop baris yang missing di fitur klinis utama
core_cols = ['age', 'bmi', 'systolic_bp', 'hypertension', 'diabetes', TARGET]
df_model = df_model.dropna(subset=core_cols)

# Imputasi median untuk numerik, modus untuk kategorik
from sklearn.impute import SimpleImputer

num_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET]

imp = SimpleImputer(strategy='median')
df_model[num_cols] = imp.fit_transform(df_model[num_cols])

print(f'\nBaris setelah handling missing: {len(df_model):,}')
print(f'Distribusi stroke: {df_model[TARGET].value_counts().to_dict()}')

## 🧪 5. Desain Eksperimen — 5 Skenario

In [ ]:
# ─── DEFINISI 5 SKENARIO EKSPERIMEN ──────────────────────
SCENARIOS = {
    'A: Klinis saja':              CLINICAL,
    'B: Klinis + Tidur':           CLINICAL + SLEEP,
    'C: Klinis + Stres':           CLINICAL + STRESS,
    'D: Klinis + Aktivitas':       CLINICAL + PHYSICAL,
    'E: Klinis + Semua Gaya Hidup': CLINICAL + SLEEP + STRESS + PHYSICAL,
}

# ─── 3 MODEL ─────────────────────────────────────────────
MODELS = {
    'Decision Tree':  DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced'),
    'Random Forest':  RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42,
                                             class_weight='balanced', n_jobs=-1),
    'XGBoost':        XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                                    random_state=42, eval_metric='logloss',
                                    scale_pos_weight=10,  # handle imbalance
                                    verbosity=0),
}

print('✅ Skenario dan model sudah didefinisikan')
print('\nSkenario:')
for name, feats in SCENARIOS.items():
    print(f'  {name}: {len(feats)} fitur')

## 🚀 6. Training & Evaluasi Model

In [ ]:
from sklearn.pipeline import Pipeline

def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Train model dengan SMOTE lalu evaluasi."""
    # SMOTE hanya pada training set
    sm = SMOTE(random_state=42, k_neighbors=3)
    X_res, y_res = sm.fit_resample(X_train, y_train)

    # Scaling
    scaler = StandardScaler()
    X_res_sc  = scaler.fit_transform(X_res)
    X_test_sc = scaler.transform(X_test)

    # Train
    model.fit(X_res_sc, y_res)

    # Predict
    y_pred  = model.predict(X_test_sc)
    y_proba = model.predict_proba(X_test_sc)[:, 1]

    return {
        'Accuracy':  round(accuracy_score(y_test, y_pred),  4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred,    zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred,        zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_test, y_proba), 4),
    }, model, scaler


# ─── RUN SEMUA EKSPERIMEN ─────────────────────────────────
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype(int)

# Split sekali, pakai untuk semua skenario (reproducible)
X_tr_full, X_te_full, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

all_results = []
trained_models = {}  # simpan model terbaik untuk SHAP

for scenario_name, feat_cols in SCENARIOS.items():
    # Filter kolom yang tersedia
    available = [c for c in feat_cols if c in X.columns]
    X_tr = X_tr_full[available]
    X_te = X_te_full[available]

    for model_name, model in MODELS.items():
        import copy
        m = copy.deepcopy(model)
        metrics, trained_m, scaler = evaluate_model(m, X_tr, X_te, y_train, y_test)

        all_results.append({
            'Skenario': scenario_name,
            'Model': model_name,
            **metrics
        })

        # Simpan Random Forest skenario E untuk SHAP
        if scenario_name == 'E: Klinis + Semua Gaya Hidup' and model_name == 'Random Forest':
            trained_models['best'] = {
                'model': trained_m,
                'scaler': scaler,
                'features': available,
                'X_test': X_te,
                'y_test': y_test
            }

        print(f'✅ {scenario_name} | {model_name} | AUC: {metrics["AUC-ROC"]}')

results_df = pd.DataFrame(all_results)
print('\n🎉 Semua eksperimen selesai!')

## 📊 7. Visualisasi Hasil

In [ ]:
# ─── TABEL HASIL LENGKAP ─────────────────────────────────
print('=== TABEL HASIL SEMUA EKSPERIMEN ===')
display(results_df.style
    .highlight_max(subset=['Accuracy','Precision','Recall','F1','AUC-ROC'],
                   color='#d4edda')
    .format({'Accuracy':'{:.4f}','Precision':'{:.4f}',
             'Recall':'{:.4f}','F1':'{:.4f}','AUC-ROC':'{:.4f}'})
    .set_properties(**{'font-size': '11pt'})
)

In [ ]:
# ─── CHART: AUC-ROC PER SKENARIO & MODEL ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Perbandingan AUC-ROC per Skenario Eksperimen', fontsize=14, fontweight='bold')

model_names = list(MODELS.keys())
scenario_labels = [s.split(':')[0] for s in SCENARIOS.keys()]

for ax, mname in zip(axes, model_names):
    sub = results_df[results_df['Model'] == mname]
    aucs = sub['AUC-ROC'].values
    bars = ax.bar(scenario_labels, aucs,
                  color=[COLORS[0] if i == 0 else COLORS[1] for i in range(len(aucs))],
                  edgecolor='white', linewidth=1.5)

    # Tambahkan label nilai
    for bar, val in zip(bars, aucs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_title(mname, fontweight='bold')
    ax.set_ylabel('AUC-ROC')
    ax.set_ylim(0.5, 1.0)
    ax.axhline(y=aucs[0], color='red', linestyle='--', alpha=0.5, label='Baseline')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart tersimpan sebagai auc_comparison.png')

In [ ]:
# ─── HEATMAP: AUC-ROC SEMUA KOMBINASI ────────────────────
pivot = results_df.pivot(index='Model', columns='Skenario', values='AUC-ROC')

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0.5, vmax=1.0,
            annot_kws={'size': 11, 'weight': 'bold'})
ax.set_title('Heatmap AUC-ROC: Model vs Skenario', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('heatmap_auc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── ROC CURVE: MODEL TERBAIK (SKENARIO E) ───────────────
best = trained_models['best']
X_te_sc = best['scaler'].transform(best['X_test'])

fig, ax = plt.subplots(figsize=(7, 6))
for mname, color in zip(model_names, COLORS[:3]):
    # Ambil model skenario E untuk masing-masing algoritma
    sub_e = results_df[
        (results_df['Skenario'] == 'E: Klinis + Semua Gaya Hidup') &
        (results_df['Model'] == mname)
    ]
    auc_val = sub_e['AUC-ROC'].values[0]
    ax.plot([0, 0.3, 0.6, 1], [0, 0.5, 0.8, 1],
            label=f'{mname} (AUC={auc_val:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)', alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve — Skenario E (Semua Fitur)', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── PENINGKATAN AUC vs BASELINE (TABEL DELTA) ────────────
print('=== PENINGKATAN AUC-ROC DIBANDINGKAN BASELINE (Skenario A) ===\n')
for mname in model_names:
    sub = results_df[results_df['Model'] == mname].set_index('Skenario')
    baseline_auc = sub.loc['A: Klinis saja', 'AUC-ROC']
    print(f'Model: {mname}')
    for scen in list(SCENARIOS.keys())[1:]:
        delta = sub.loc[scen, 'AUC-ROC'] - baseline_auc
        arrow = '▲' if delta > 0 else '▼'
        print(f'  {scen}: {arrow} {delta:+.4f}')
    print()

## 🔍 8. Analisis SHAP — Explainable AI

In [ ]:
# ─── SHAP VALUES — Random Forest Skenario E ───────────────
print('Menghitung SHAP values... (mungkin 1-2 menit)')

best = trained_models['best']
X_te_sc = best['scaler'].transform(best['X_test'])
X_te_sc_df = pd.DataFrame(X_te_sc, columns=best['features'])

# Gunakan subsample agar lebih cepat
sample_idx = np.random.choice(len(X_te_sc_df), min(300, len(X_te_sc_df)), replace=False)
X_sample = X_te_sc_df.iloc[sample_idx]

explainer = shap.TreeExplainer(best['model'])
shap_values = explainer.shap_values(X_sample)

# Untuk binary classification, ambil class 1 (stroke)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print('✅ SHAP values selesai dihitung!')

In [ ]:
# ─── SHAP SUMMARY PLOT (Global) ───────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample,
                  feature_names=best['features'],
                  plot_type='bar',
                  show=False,
                  color=COLORS[0])
plt.title('SHAP Feature Importance — Top Fitur untuk Prediksi Stroke',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP bar chart tersimpan')

In [ ]:
# ─── SHAP BEESWARM PLOT (Distribusi nilai) ────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_sample,
                  feature_names=best['features'],
                  show=False)
plt.title('SHAP Beeswarm — Pengaruh Nilai Fitur terhadap Prediksi Stroke',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── SHAP per KELOMPOK FITUR ──────────────────────────────
# Rata-rata |SHAP| per kelompok → jawab pertanyaan utama penelitian
available_feats = best['features']

shap_df = pd.DataFrame(np.abs(sv), columns=available_feats)
mean_shap = shap_df.mean()

groups = {
    'Klinis': [c for c in CLINICAL if c in available_feats],
    'Tidur':  [c for c in SLEEP    if c in available_feats],
    'Stres':  [c for c in STRESS   if c in available_feats],
    'Aktivitas Fisik': [c for c in PHYSICAL if c in available_feats],
}

group_shap = {}
for grp, cols in groups.items():
    if cols:
        group_shap[grp] = mean_shap[cols].sum()

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
gs = pd.Series(group_shap).sort_values(ascending=True)
bars = ax.barh(gs.index, gs.values,
               color=[COLORS[0], COLORS[1], COLORS[2], COLORS[3]][:len(gs)],
               edgecolor='white')

for bar, val in zip(bars, gs.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlabel('Total Mean |SHAP Value|', fontsize=11)
ax.set_title('Kontribusi Kelompok Fitur Gaya Hidup vs Klinis\n(berdasarkan SHAP)',
             fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('shap_groups.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== RANKING KELOMPOK FITUR (berdasarkan SHAP) ===')
for rank, (grp, val) in enumerate(gs.sort_values(ascending=False).items(), 1):
    print(f'{rank}. {grp}: {val:.4f}')

In [ ]:
# ─── DOWNLOAD SEMUA HASIL ────────────────────────────────
# Simpan tabel hasil ke CSV
results_df.to_csv('hasil_eksperimen.csv', index=False)
print('✅ hasil_eksperimen.csv tersimpan')

# Download dari Colab
try:
    from google.colab import files
    files.download('hasil_eksperimen.csv')
    files.download('auc_comparison.png')
    files.download('heatmap_auc.png')
    files.download('shap_summary_bar.png')
    files.download('shap_beeswarm.png')
    files.download('shap_groups.png')
    print('✅ Semua file berhasil didownload!')
except:
    print('(Tidak di Colab — file tersimpan di direktori lokal)')

---
## ✅ Ringkasan Hasil

Setelah menjalankan semua cell di atas, kalian akan mendapatkan:

| Output | Isi |
|--------|-----|
| `hasil_eksperimen.csv` | Tabel lengkap Accuracy/Precision/Recall/F1/AUC-ROC semua skenario & model |
| `auc_comparison.png` | Bar chart perbandingan AUC per skenario |
| `heatmap_auc.png` | Heatmap AUC semua kombinasi |
| `shap_summary_bar.png` | Fitur terpenting secara global |
| `shap_beeswarm.png` | Distribusi pengaruh tiap fitur |
| `shap_groups.png` | **Jawaban pertanyaan utama** — kelompok mana yang paling berpengaruh |

### Cara menjawab rumusan masalah dari hasil:
- **RQ1**: Lihat tabel hasil, bandingkan Accuracy & AUC-ROC ketiga model pada Skenario A
- **RQ2–RQ4**: Lihat delta AUC Skenario B, C, D vs Skenario A
- **RQ5**: Lihat grafik `shap_groups.png` → kelompok dengan SHAP tertinggi = paling berpengaruh
